In [1]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.12.12 (main, Jan 14 2026, 19:35:58) [Clang 21.1.4 ]


In [2]:
%pip install -U ultralytics

/home/jyothish/amrit/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from pathlib import Path

import torch
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)

Ultralytics version: 8.4.143
PyTorch version: 2.6.0+cu124


In [4]:
from ultralytics import YOLO

print("YOLO class imported successfully.")

YOLO class imported successfully.


In [5]:
import zipfile
from pathlib import Path

zip_path = Path("/home/jyothish/amrit/amrittt/Checking/processed.zip")
extract_path = Path("/home/jyothish/amrit/amrittt/Checking")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted.")

Dataset extracted.


In [6]:
base = Path("/home/jyothish/amrit/amrittt/Checking/processed")

for split in ["train", "valid", "test"]:
    image_dir = base / split / "images"
    label_dir = base / split / "labels"

    print(f"\n{split.upper()}")
    print("Images:", len(list(image_dir.glob("*"))))
    print("Labels:", len(list(label_dir.glob("*.txt"))))


TRAIN
Images: 6254
Labels: 6254

VALID
Images: 1784
Labels: 1784

TEST
Images: 901
Labels: 901


In [7]:
yaml_content = """
path: /home/jyothish/amrit/amrittt/Checking/processed

train: train/images
val: valid/images
test: test/images

names:
  0: fire
  1: other
  2: smoke
"""

Path("/home/jyothish/amrit/amrittt/Checking/processed/data.yaml").write_text(yaml_content)

print(Path("/home/jyothish/amrit/amrittt/Checking/processed/data.yaml").read_text())


path: /home/jyothish/amrit/amrittt/Checking/processed

train: train/images
val: valid/images
test: test/images

names:
  0: fire
  1: other
  2: smoke



In [8]:
import yaml

yaml_path = "/home/jyothish/amrit/amrittt/Checking/processed/data.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

print(data)

{'path': '/home/jyothish/amrit/amrittt/Checking/processed', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'fire', 1: 'other', 2: 'smoke'}}


In [9]:
image_dir = Path("/home/jyothish/amrit/amrittt/Checking/processed/train/images")
label_dir = Path("/home/jyothish/amrit/amrittt/Checking/processed/train/labels")

image_files = list(image_dir.glob("*"))

sample_image = image_files[0]
sample_label = label_dir / f"{sample_image.stem}.txt"

print("Image:", sample_image)
print("Label:", sample_label)
print("\nLabel contents:")
print(sample_label.read_text())

Image: /home/jyothish/amrit/amrittt/Checking/processed/train/images/middle_-2348-_jpg.rf.03184b556f80db7cc7352d6b68888a85.jpg
Label: /home/jyothish/amrit/amrittt/Checking/processed/train/labels/middle_-2348-_jpg.rf.03184b556f80db7cc7352d6b68888a85.txt

Label contents:
0 0.12860576923076922 0.7908653846153846 0.15504807692307693 0.10817307692307693
0 0.2512019230769231 0.5324519230769231 0.09254807692307693 0.10336538461538461
0 0.5432692307692307 0.6766826923076923 0.08774038461538461 0.13942307692307693
0 0.4206730769230769 0.7524038461538461 0.040865384615384616 0.052884615384615384
0 0.6959134615384616 0.7836538461538461 0.06129807692307692 0.06009615384615385
0 0.35096153846153844 0.42908653846153844 0.036057692307692304 0.05048076923076923
2 0.1622596153846154 0.18509615384615385 0.31971153846153844 0.36899038461538464
2 0.11298076923076923 0.5384615384615384 0.18028846153846154 0.3137019230769231
2 0.30649038461538464 0.7211538461538461 0.19951923076923078 0.27163461538461536
2 0

In [10]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Dataset paths
BASE_DIR = Path("/home/jyothish/amrit/amrittt/Checking/processed")

IMAGE_DIR = BASE_DIR / "train" / "images"
LABEL_DIR = BASE_DIR / "train" / "labels"

# Class mapping
CLASS_NAMES = {
    0: "fire",
    1: "other",
    2: "smoke"
}

# Get images
image_files = sorted([
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
])

print("Total training images:", len(image_files))

Total training images: 6254


In [11]:
def get_images(split):
    image_dir = BASE_DIR / split / "images"

    return sorted([
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ])


def show_dataset_image(split, index):

    images = get_images(split)

    image_path = images[index]

    label_dir = BASE_DIR / split / "labels"
    label_path = label_dir / f"{image_path.stem}.txt"

    image = cv2.imread(str(image_path))

    if image is None:
        print("Could not read:", image_path)
        return

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    annotated_image = image.copy()

    annotations = []

    if label_path.exists():

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])

            x_center = float(parts[1]) * width
            y_center = float(parts[2]) * height
            box_width = float(parts[3]) * width
            box_height = float(parts[4]) * height

            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)

            class_name = CLASS_NAMES.get(
                class_id,
                f"unknown_{class_id}"
            )

            annotations.append(
                f"{class_id} → {class_name}"
            )

            cv2.rectangle(
                annotated_image,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2
            )

            cv2.putText(
                annotated_image,
                class_name,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

    # Display image
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_image)
    plt.axis("off")
    plt.title(
        f"{split.upper()} | "
        f"{index + 1}/{len(images)} | "
        f"{image_path.name}"
    )
    plt.show()

    # Display label information
    print("=" * 70)
    print("IMAGE:", image_path.name)
    print("LABEL:", label_path.name)
    print("=" * 70)

    if label_path.exists():
        print(label_path.read_text())
    else:
        print("Label file not found!")

    print("\nCLASS INFORMATION")
    print("-" * 70)

    if annotations:
        for annotation in annotations:
            print(annotation)
    else:
        print("No annotations")

In [12]:
split_dropdown = widgets.Dropdown(
    options=["train", "valid", "test"],
    value="train",
    description="Split:"
)

index_slider = widgets.IntSlider(
    min=0,
    max=len(get_images("train")) - 1,
    step=1,
    value=0,
    description="Image:"
)


def update_slider(*args):

    split = split_dropdown.value
    images = get_images(split)

    index_slider.max = len(images) - 1
    index_slider.value = min(index_slider.value, len(images) - 1)


split_dropdown.observe(update_slider, names="value")


def interactive_viewer(split, index):

    show_dataset_image(split, index)


widgets.interact(
    interactive_viewer,
    split=split_dropdown,
    index=index_slider
);

interactive(children=(Dropdown(description='Split:', options=('train', 'valid', 'test'), value='train'), IntSl…

In [13]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA A10


In [14]:
model = YOLO("yolo26n.pt")

print("Model loaded successfully!")

Model loaded successfully!


In [18]:
model.train(
    data="/home/jyothish/amrit/amrittt/Checking/processed/data.yaml",
    epochs=1,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project="/home/jyothish/amrit/amrittt/Checking/runs",
    name="gpu_smoke_test"
)

New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.143 🚀 Python-3.12.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A10, 22516MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jyothish/amrit/amrittt/Checking/processed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x739026a65a30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04